# Phase 1 full pipeline, Drive mode

This notebook runs **`lexical_gap_full_pipeline.ipynb` unchanged**, with every output written to Google Drive. Open that notebook to read the code; it embeds every source file.

1. *Runtime > Change runtime type > T4 GPU*. Only the dense step needs the GPU.
2. *Runtime > Run all*. Allow Drive access when asked. It takes about 1-1.5 h. At the end, `report.pdf` and `results.zip` are also downloaded to your computer.

**How it works.**
- It builds `/content/proj` with `results/`, `cache/`, `figures/` and `report/` linked into `MyDrive/IR_Lexical_Gap/project/`, and `data/` linked to `MyDrive/IR_Lexical_Gap/data`. The data is filled by `00_SETUP_drive`; if it is there, the download step is skipped.
- It then executes the code cells of `01_notebooks/lexical_gap_full_pipeline.ipynb` one after another, in this kernel.
- `phase2_analysis` needs these outputs, in particular `project/results/*.csv` and `project/cache/runs/*.pkl` (about 2 GB).

**If it stops** (error or disconnect): fix the cause, set `START` in the loop cell (the second-to-last code cell) to the cell number printed in the error, and run that cell again. Re-run the mount and link cells first if the runtime was reset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
shutil.copy('/content/drive/MyDrive/IR_Lexical_Gap/02_code/phase2/setup_drive.py', '/content/setup_drive.py')  # Drive copy of phase2/colab/setup_drive.py

In [ ]:
!python /content/setup_drive.py link
%cd /content/proj

In [ ]:
import json
from IPython import get_ipython
NB = '/content/drive/MyDrive/IR_Lexical_Gap/01_notebooks/lexical_gap_full_pipeline.ipynb'
START = 0  # code-cell index to (re)start from
cells = [c for c in json.load(open(NB))['cells'] if c['cell_type'] == 'code']
for i in range(START, len(cells)):
    src = cells[i]['source']
    src = ''.join(src) if isinstance(src, list) else src
    print(f'\n===== Phase-1 code cell {i}/{len(cells) - 1}: {src.strip().splitlines()[0][:90]}', flush=True)
    res = get_ipython().run_cell(src)
    if not res.success:
        raise RuntimeError(f'Phase-1 code cell {i} failed. Fix the cause, set START = {i} and run this cell again.')
print('\nPhase 1 finished: outputs in MyDrive/IR_Lexical_Gap/project/{results,cache,figures,report}')

In [ ]:
import glob, shutil
for f in glob.glob('/content/proj/src/*.py'):
    shutil.copy(f, '/content/drive/MyDrive/IR_Lexical_Gap/project/src/')
shutil.copy('/content/proj/report/report.pdf', '/content/drive/MyDrive/IR_Lexical_Gap/05_reports_and_paper/report_regenerated.pdf')
from google.colab import drive
drive.flush_and_unmount()
print('Drive flushed. Next: phase2_dense_runs (T4), phase2_ce_rerank (T4), phase2_analysis (CPU).')